In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# Functions

# A function that flattens the lower triangle of a numpy array in column-major order, 
# optionally excluding the diagonal.
def flatten_lower_triangle(array, exclude_diagonal=False):
    if exclude_diagonal:
        return array[np.tril_indices(array.shape[0], -1)]
    else:
        return array[np.tril_indices(array.shape[0])]

def generate_null_distribution(expression_flat, proximity_flat, num_samples=1000):
    assert len(expression_flat) == len(proximity_flat)
    null = np.zeros(len(expression_flat))
    expression_flat_copy = expression_flat.copy()

    for i in range(num_samples):
        np.random.shuffle(expression_flat_copy)
        null += np.multiply(expression_flat_copy, proximity_flat)
        
    return null / num_samples


In [ ]:
# Tests for functions

def test_flatten_lower_triangle():
    # Test that the function returns the correct values for a 3x3 matrix
    test_matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
    test_output = flatten_lower_triangle(test_matrix)
    expected_output = np.array([1, 4, 5, 7, 8, 9])
    assert np.array_equal(test_output, expected_output), f"Expected {expected_output}, but got {test_output}"
    
    # Test that the function returns the correct values for a 4x4 matrix
    test_matrix = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]])
    test_output = flatten_lower_triangle(test_matrix)
    expected_output = np.array([1, 5, 6, 9, 10, 11, 13, 14, 15, 16])
    assert np.array_equal(test_output, expected_output), f"Expected {expected_output}, but got {test_output}"
    
    # Test that the function returns the correct values for a 5x5 matrix
    test_matrix = np.array([[1, 2, 3, 4, 5], [6, 7, 8, 9, 10], [11, 12, 13, 14, 15], [16, 17, 18, 19, 20], [21, 22, 23, 24, 25]])
    test_output = flatten_lower_triangle(test_matrix)
    expected_output = np.array([1, 6, 7, 11, 12, 13, 16, 17, 18, 19, 21, 22, 23, 24, 25])
    assert np.array_equal(test_output, expected_output), f"Expected {expected_output}, but got {test_output}"
    
    # Test that the function returns the correct values for a 3x3 matrix with the diagonal excluded
    test_matrix = np.array([[1, 2, 3], [4, 5, 6], [7, 8, 9]])
    test_output = flatten_lower_triangle(test_matrix, exclude_diagonal=True)
    expected_output = np.array([4, 7, 8])
    assert np.array_equal(test_output, expected_output), f"Expected {expected_output}, but got {test_output}"
    
	# Test that the function returns the correct values for a 4x4 matrix with the diagonal excluded
    test_matrix = np.array([[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]])
    test_output = flatten_lower_triangle(test_matrix, exclude_diagonal=True)
    expected_output = np.array([5, 9, 10, 13, 14, 15])
    assert np.array_equal(test_output, expected_output), f"Expected {expected_output}, but got {test_output}"
    
# Run tests
test_flatten_lower_triangle()
print("All tests passed successfully")

# Relationship between gene expression across zooids and physical proximity of genes in the siphonophore *Nanomia*

The goal of this analysis is to see if there are genes with correlated differential expression across zooids that also have close proximity in the genome. We will see if any such relationship is higher than would be expected by chance, and finally drill down a bit on the specifics of the correlation.

## Data import

First, we import the expression data. These were generated in R by:
- Creating a normalized count matrix with DESeq, where rows are genes and columns are samples
- Averaging the normalized counts for each combination of tissue and stage that was sequenced, eg `gastrozood_mature`

Specifically:

```R
D = DESeq2::counts( de_stage_tissue, normalized=TRUE)
table(de_stage_tissue$tissue, de_stage_tissue$stage)

sample_info <- data.frame(tissue = de_stage_tissue$tissue, stage = de_stage_tissue$stage)
group <- with(sample_info, paste(tissue, stage, sep = "_"))

# Initialize an empty list to store average counts for each group
average_counts <- list()

# Loop through each unique group to calculate mean counts
for(g in unique(group)) {
  # Find columns that belong to this group
  cols <- which(group == g)
  
  # Calculate the mean across these columns and store it
  average_counts[[g]] <- rowMeans(D[, cols, drop = FALSE])
}

# Combine the results into a new matrix
average_counts_matrix <- do.call(cbind, average_counts)
# Optionally, name the columns after the groups
colnames(average_counts_matrix) <- names(average_counts)

# Convert the matrix to a data frame
average_counts_df <- as.data.frame(average_counts_matrix)

# Add the row names of the matrix as the first column in the data frame
average_counts_df$gene <- rownames(average_counts_matrix)

# To ensure 'gene' is the first column, reorder the columns
average_counts_df <- average_counts_df[, c("gene", setdiff(names(average_counts_df), "gene"))]
write.table(average_counts_df, file = "average_counts_df.tsv", sep = "\t", quote = FALSE, row.names = FALSE)
```

In [ ]:
# Import the expression data

expression_df = pd.read_csv("average_counts_df.tsv", sep="\t")
#expression_df = expression_df.drop(columns=['gonodendron_male_mature'])
#print(expression_df.head())
# Remove -RA from gene names
expression_df['gene'] = expression_df['gene'].str.replace('-RA', '')

# Extract gene names
gene_names_expression = expression_df['gene'].values.tolist()

# Make sure there are no duplicated gene IDs
assert len(gene_names_expression) == len(set(gene_names_expression))

# Extract numerical data for analysis
numerical_data = expression_df.drop(columns=['gene']).values

# Treatment names are the column names excluding the first column 'gene'
treatment_names = expression_df.columns[1:].tolist()

expression = np.array(numerical_data)

expression.shape

Now we import the gff data, which gives gen positions. 

There is some preprocessing:
- We consider only the chromosomes, and ignore small scaffolds.
- Genes are sorted by chromosome and then start position.


In [ ]:
gff_file = "PO2744_Nanomia_bijuga.annotation.gff"
gff_data = pd.read_csv(gff_file, sep="\t", comment="#", header=None)
gff_data.columns = ["seqid", "source", "type", "start", "end", "score", "strand", "phase", "attributes"]

# Extract gene features
gff_data = gff_data[gff_data["type"] == "gene"]

# Retain only chromosome annotations, ie where seqid contains _LG
gff_data = gff_data[gff_data["seqid"].str.contains("_LG")]

# Sort gene data by seqid and then by start position
gff_data = gff_data.sort_values(["seqid", "start"])

# Here is an example of an attribute field:
# ID=ANN00001;Note=Protein of unknown function;
# Extract the gene ID and the gene note from the attributes column and add them as separate columns
gff_data["gene"] = gff_data["attributes"].str.extract(r"ID=([^;]+);")

# Make sure there are no duplicated gene IDs
assert gff_data["gene"].is_unique

# Make sure index is sequential
gff_data = gff_data.reset_index(drop=True)

In [ ]:
n_genes = gff_data.shape[0]
print(f"Number of genes: {n_genes}")

In [ ]:
# Print the first 5 genes
print(gff_data.head())

In [ ]:
print("First 5 expression gene names: ", gene_names_expression[:5])
print("first 5 gff gene names: ", gff_data["gene"].values[:5])

Subset down to the intersection of genes that are present in both the expression data and the gff data. Put the expression data in the same order as the gff data, ie we are going to keep everything ordered by physical position

In [ ]:
# Subset both expression and gff down to the same set of genes

print(f"Original number of gff genes: {len(gff_data["gene"])}")
print(f"Original number of expression genes: {len(gene_names_expression)}")

# Get the set of intersecting genes by converting the gene names to sets and taking the intersection
gene_names_gff_set = set(gff_data["gene"].values)
gene_names_expression_set = set(gene_names_expression)
common_genes = gene_names_gff_set.intersection(gene_names_expression_set)

print(f"Number of common genes: {len(common_genes)}")

# find the elements of gene_names_expression that are in common_genes
expression_common_genes = np.array([gene in common_genes for gene in gene_names_expression])

print(f"Number of common genes in expression: {np.sum(expression_common_genes)}")

gene_names_expression = np.array(gene_names_expression)[expression_common_genes].tolist()

# find the elements of gene_names_gff that are in common_genes
gff_common_genes = np.array([gene in common_genes for gene in gff_data["gene"].values])
gff_data = gff_data[gff_common_genes]


print(f"Final number of gff genes: {len(gff_data["gene"])}")
print(f"Final number of expression genes: {len(gene_names_expression)}")

In [ ]:
gff_gene_order = gff_data['gene'].tolist()

new_index = [gene_names_expression.index(gene) for gene in gff_gene_order]

reordered_gene_names_expression = [gene_names_expression[i] for i in new_index]
gene_names_expression = reordered_gene_names_expression

expression = expression[new_index]




In [ ]:
# Check that the gene names are in the same order

gene_names_gff = gff_data["gene"].values

print(np.all(gene_names_gff == gene_names_expression))

# Print number of genes that don't match
print(np.sum(gene_names_gff != gene_names_expression))

# If there are any genes that don't match, print the first 5
print(gene_names_gff[gene_names_gff != gene_names_expression][:5])


In [ ]:
assert np.all(gene_names_gff == gene_names_expression)

In [ ]:
print(f"gff names: {gff_data["gene"].values[0:5]}")
print(f"expression names: {gene_names_expression[0:5]}")

# print cases where the gene names are not the same
print(gff_data["gene"].values[gff_data["gene"].values != gene_names_expression])


## Create matrices of physical proximity and expression correlation

Create a physical distance matrix, where:
- distance is `nan` if the genes are on different chromosomes
- distance is the number of genes away if the genes are on the same chromosome. eg 0 for self, 1 for adjacent, etc.

In [ ]:
# Create a DataFrame with a MultiIndex
df = pd.DataFrame(gff_data["seqid"])
df.index = pd.MultiIndex.from_arrays([df["seqid"], df.index])

# Initialize the gene_by_distance matrix
gene_by_distance = np.full((n_genes, n_genes), np.nan)

# Fill the matrix
for seqid, group in df.groupby(level=0):
    indices = group.index.get_level_values(1)
    distances = np.abs(indices.values[:, None] - indices.values)
    gene_by_distance[np.ix_(indices, indices)] = distances

In [ ]:
print(gene_by_distance[0,:])

Create a physical proximity matrix with values of 0-1, where:
- 0 if the genes are on different chromosomes
- 1 on the diagonal, ie proximity to self
- A gaussian kernal function of the distance between the genes on the same chromosome. This is to give a smooth transition from 1 to 0 as the distance increases. The width of the kernel is tuned by specifying the distince (in terms of the number of genes) that results in a value of 0.5.

In [ ]:
# From the distance matrix, construct a proximity matrix where the elements are 1 if the corresponding genes are adjacent and 0 if they are really far apart. 
# Don't use actual distances from the gff, measure distance in terms of number of genes between two genes.
# Use a gaussian kernel to determine how close two genes are.

# Define the kernel
def gaussian_kernel(x, sigma=1):
	return np.exp(-x**2 / (2 * sigma**2))

# Calculate the sigma needed for the kernal to give a value of 0.5 when the distance is n
def calculate_sigma(n):
	return np.sqrt(-n**2 / (2 * np.log(0.5)))

# Apply the kernel to the distance matrix
proximity_matrix = gaussian_kernel(gene_by_distance, calculate_sigma(50))   #50 is abstract number/randomly chosen and need to maybe look at gene densities across chromosomes 

# Replace nan values with 0
proximity_matrix = np.nan_to_num(proximity_matrix)



In [ ]:
print(proximity_matrix[0,:])

In [ ]:
# plot the proximity matrix
plt.imshow(proximity_matrix, cmap="viridis")
plt.colorbar()
plt.show()

In [ ]:
n = 500
# Plot the first n rows and columns of the proximity matrix
plt.imshow(proximity_matrix[:n, :n], cmap="viridis")
plt.colorbar()
plt.show()

## Subset matrices based on expression magnitude and variance

Take a look at the expression data. We want the analysis to focus on the subset of genes that:
- Have a relatively high expression level. Differences among genes with very low expression levels are likely to be noisy.
- Have a relatively high variance in expression level across tissue_stage combinations. This will put the focus on genes that are differentially expressed across tissues and stages.

In [ ]:
print(expression)

In [ ]:
#view the expression levels per each column (zooid type)
#build a histogram for each column in the expression matrix
#get maximum expression value for each column in the expression matrix
max_expression = np.max(expression, axis=0)
#calculate mean, and median for each expression column
#for i in range(expression.shape[1]):
    #print(treatment_names[i], np.mean(expression[:, i]), np.median(expression[:, i]), np.max(expression[:, i]))
#count the number of genes that have higher expression than 1000, 2000, 5000 for each tretment
for i in range(expression.shape[1]):
    print(treatment_names[i], np.sum(expression[:, i] > 1000))
for i in range(expression.shape[1]):
    print(treatment_names[i], np.sum(expression[:, i] > 2000))
for i in range(expression.shape[1]):
    print(treatment_names[i], np.sum(expression[:, i] > 5000))

In [ ]:
# Find the genes that have the most variable expression across all treatments

# Assuming 'expression_matrix' is your gene expression matrix
# Calculate the mean and standard deviation for each gene across tissues
gene_means = np.mean(expression, axis=1)
gene_std_devs = np.std(expression, axis=1)

# Calculate the coefficient of variation (CV) for each gene
gene_cvs = gene_std_devs / gene_means

# Replace NaN values with 0
gene_cvs = np.nan_to_num(gene_cvs)

# Identify the gene(s) with the highest CV
# This gives the index of the gene with the highest CV
max_cv_gene_index = np.argmax(gene_cvs)

print(f"Gene with the highest coefficient of variation across tissues: Gene {max_cv_gene_index}")
print(f"Coefficient of variation of this gene's expression across tissues: {gene_cvs[max_cv_gene_index]}")

# If you want to find more than one gene, you can sort the genes by their CV
sorted_gene_indices_by_cv = np.argsort(gene_cvs)[::-1]  # Descending order

print("Genes ranked by their coefficient of variation across tissues (from most to least):")
print(sorted_gene_indices_by_cv)


In [ ]:
# Print a histogram of the coefficient of variation values


plt.hist(gene_cvs, bins=50, color='skyblue', edgecolor='black')
plt.xlabel('Coefficient of Variation')
plt.ylabel('Frequency')
plt.title('Distribution of Coefficients of Variation')
plt.show()

In [ ]:
# Calculate the max for each gene and plot a histogram

range_max = 2000
max_expression = np.max(expression, axis=1)


plt.hist(max_expression, bins=50, range=(0, range_max), color='skyblue', edgecolor='black')
plt.xlabel('Max Expression Value')
plt.ylabel('Frequency')
plt.title('Distribution of Max Expression Values')
plt.show()

In [ ]:
# Plot a histogram of the mean expression values

plt.hist(gene_means, bins=50, range=(0, range_max), color='skyblue', edgecolor='black')
plt.xlabel('Mean Expression Value')
plt.ylabel('Frequency')
plt.title('Distribution of Mean Expression Values')
plt.show()


In [ ]:
max_expression_threshold = 200
cv_threshold = 0.5

In [ ]:
# Subset the expression data to include only the genes with max expression greater than max_expression_threshold

# Find the indices of genes with max expression greater than max_expression_threshold
high_expression_genes = np.where(max_expression > max_expression_threshold)[0]

# Subset the expression matrix to include only the high expression genes
high_expression_data = expression[high_expression_genes, :]

# Plot the CV distribution for the high expression genes
high_expression_gene_cvs = gene_cvs[high_expression_genes]

plt.hist(high_expression_gene_cvs, bins=50, color='skyblue', edgecolor='black')
plt.xlabel('Coefficient of Variation')
plt.ylabel('Frequency')
plt.title('Distribution of Coefficients of Variation for High Expression Genes')
plt.show()

In [ ]:
# Get a list of the gene names for the genes with max expression greater than 1000 and CV greater than cv_threshold

# Find the indices of genes that meet the criteria
selected_genes = np.where((max_expression > max_expression_threshold) & (gene_cvs > cv_threshold))[0]

# Get the gene names for the selected genes
selected_gene_names = np.array(gene_names_expression)[selected_genes].tolist()

#print(f"Genes with max expression > {max_expression_threshold} and CV > {cv_threshold}:")
#print(selected_gene_names)

In [ ]:
expression_similarity_matrix = np.corrcoef(expression)
print(expression_similarity_matrix)

In [ ]:
focal_proximity_matrix = proximity_matrix[np.ix_(selected_genes, selected_genes)]
focal_expression_similarity_matrix = expression_similarity_matrix[np.ix_(selected_genes, selected_genes)]

In [ ]:
print(focal_proximity_matrix.shape)
print(focal_expression_similarity_matrix.shape)

## Examine the global relationship between physical proximity and expression correlation

Now we are ready to compare physical proximity to expression correlation. We will consider only the values below the diagonal, as the matrices are symmetric and the diagonal has high values we are uninterested in. We therefore flatten all the elements below the diagonal into a 1D array for each matrix (proximity and expression). We then take the product of the two arrays. Each element will be:
- Close to 1 only if the gene pair has high expression correlation and high physical proximity
- Close to 0 if the gene pair has low expression correlation and/or low physical proximity

We can then generate a null distribution by shuffling the expression data and repeating the above steps. This will give us a sense of how likely it is to see the observed a strong relationship (ie values close to 1) between expression and proximity by chance.

In [ ]:
# Flatten focal proximity matrix
focal_proximity_matrix_flat = flatten_lower_triangle(focal_proximity_matrix, exclude_diagonal=True)

# Flatten focal expression similarity matrix
focal_expression_similarity_matrix_flat = flatten_lower_triangle(focal_expression_similarity_matrix, exclude_diagonal=True)

# assert that the lengths of the two flattened matrices are the same
assert len(focal_proximity_matrix_flat) == len(focal_expression_similarity_matrix_flat)

# take the element wise product of the two flattened matrices
expression_proximity_product_flat = focal_proximity_matrix_flat * focal_expression_similarity_matrix_flat

In [ ]:
# create a null distribution
null_distribution = generate_null_distribution(focal_expression_similarity_matrix_flat, focal_proximity_matrix_flat, num_samples=100)

In [ ]:
# Make a histogram of the product
plt.hist(expression_proximity_product_flat, bins=50, color='skyblue', edgecolor='black')

plt.xlabel('Product of Proximity and Expression Similarity')
plt.ylabel('Frequency')
plt.title('Distribution of Product of Proximity and Expression Similarity')
plt.show()

In [ ]:
# Plot the histogram of the product of proximity and expression similarity for the selected genes, but only from 0.5 to 1
plt.hist(expression_proximity_product_flat, bins=50, range=(0.5, 1), color='skyblue', edgecolor='black')
plt.xlabel('Product of Proximity and Expression Similarity')
plt.ylabel('Frequency')
plt.title('Distribution of Product of Proximity and Expression Similarity')
plt.show()

In [ ]:
# Plot the null distribution
plt.hist(null_distribution, bins=50, color='skyblue', edgecolor='black')
plt.xlabel('Product of Proximity and Expression Similarity')
plt.ylabel('Frequency')
plt.title('Null Distribution of Product of Proximity and Expression Similarity')
plt.show()

In [ ]:
# Plot the null distribution
plt.hist(null_distribution, bins=50, range=(0.5, 1), color='skyblue', edgecolor='black')
plt.xlabel('Product of Proximity and Expression Similarity')
plt.ylabel('Frequency')
plt.title('Null Distribution of Product of Proximity and Expression Similarity')
plt.show()

In [ ]:
# plot the histogram of the null distribution and the product of proximity and expression similarity in a single plot
# in range 0.5 to 1
plt.hist(null_distribution, bins=50, range=(0.5, 1), color='skyblue', edgecolor='black', alpha=0.5, label='Null Distribution')
plt.hist(expression_proximity_product_flat, bins=50, range=(0.5, 1), color='red', edgecolor='black', alpha=0.5, label='Product of Proximity and Expression Similarity')
plt.xlabel('Product of Proximity and Expression Similarity')
plt.ylabel('Frequency')
plt.title('Distribution of Product of Proximity and Expression Similarity')
plt.legend()
plt.show()

We have results! Main conclusions are:
- In the unshuffled data, there are quite a few genes that are both physically proximal and have high expression correlation. This is consistent with the idea that genes that are physically close are more likely to be co-regulated.
- In the shuffled data, the relationship between physical proximity and expression correlation is non-existent. This suggests that the relationship in the unshuffled data is not due to chance.

## Gene-level perspective

In [ ]:
# Create elementwise product of the two matrices
expression_proximity_product = focal_proximity_matrix * focal_expression_similarity_matrix



### Heat map of expression correlation of proximal genes

Here we sum the expression correlation $\times$ proximity product for each gene with all other genes to get a value for each gene that indicates how much its expression correlates with neighbors. 

In [ ]:
# Get the column sums of the product matrix and subtract the diagonal
expression_proximity_product_sum = np.sum(expression_proximity_product, axis=0) - np.diag(expression_proximity_product)

# Create a dataframe with selected_gene_names and expression_proximity_product_sum
gene_df = pd.DataFrame({'gene': selected_gene_names, 'expression_proximity_product_sum': expression_proximity_product_sum})
print(gene_df.shape)

# Merge the dataframe with gff_data, excluding the genes that are not in the new data frame
gene_df = pd.merge(gene_df, gff_data, on='gene')
print(gene_df.shape)

# Merge the dataframe with the expression data, excluding the genes that are not in the new data frame
gene_df = pd.merge(gene_df, expression_df, on='gene')
print(gene_df.shape)

# Add a column that has the name of the numeric column from the expression data that has the highest value for each row
gene_df['max_treatment'] = gene_df[treatment_names].idxmax(axis=1)


print(gene_df.head())

# Save the gene_df to a tsv file
gene_df.to_csv("gene_df.tsv", sep="\t", index=False)

In [ ]:
# plot a histogram of expression_proximity_product_sum
plt.hist(expression_proximity_product_sum, bins=50, color='skyblue', edgecolor='black')
plt.xlabel('Expression Proximity Product Sum')
plt.ylabel('Frequency')
plt.title('Distribution of Expression Proximity Product Sum')
plt.show()

So there are more positive relationships between expression of close genes than negative relationships.

In [ ]:
# For each unique seqid, plot a heatmap of the expression_proximity_product_sum values for the genes on that seqid

# Get the unique seqids
unique_seqids = gene_df['seqid'].unique()

# Plot a heatmap for each seqid
for seqid in unique_seqids:
    # Subset the data to include only the genes on the current seqid
    seqid_data = gene_df[gene_df['seqid'] == seqid]
    
    # Get the gene order
    gene_order = seqid_data['gene'].tolist()
    
    # Get the expression_proximity_product_sum values
    data = seqid_data['expression_proximity_product_sum'].values
    
    # Plot data as a 1D heatmap
    plt.figure(figsize=(10, 2))
    plt.imshow(data[:, None].T, cmap='viridis', aspect='auto')
    plt.yticks([])
    plt.colorbar()
    plt.title(f'Expression Proximity Product Sum for Genes on {seqid}')
    plt.show()

This shows multiple regions across the 8 chromosomes where expression is similar across zooids.

In [ ]:
product_sum_threshold = 8

# Get counts of unique max_treatment values for the subset of rows with product greater than product_sum_threshold
max_treatment_counts = gene_df[gene_df['expression_proximity_product_sum'] > product_sum_threshold]['max_treatment'].value_counts()

print(max_treatment_counts)

### Pairwise gene comparisons

Create a bedbpe file of gene pairs that are close to gether and have highly correlated expression. This can be visualized in a genome browser such as IGV

In [ ]:
product_threshold = 0.5

bedpe = pd.DataFrame(columns=['seqid1', 'start1', 'end1', 'seqid2', 'start2', 'end2', 'gene1', 'gene2', 'expression_corelation', 'proximity_expression_product'])

n_genes = len(selected_gene_names)

# Loop through each pair of genes. For each pair of genes that exceed the product_threshold, add them to the bedpe dataframe
for i in range(n_genes):
    for j in range(i + 1, n_genes):
        if expression_proximity_product[i, j] > product_threshold:
            new_row = pd.DataFrame({'seqid1': gff_data['seqid'].iloc[i], 
                                    'start1': gff_data['start'].iloc[i], 
                                    'end1': gff_data['end'].iloc[i],
                                    'seqid2': gff_data['seqid'].iloc[j], 
                                    'start2': gff_data['start'].iloc[j], 
                                    'end2': gff_data['end'].iloc[j],
                                    'gene1': selected_gene_names[i],
                                    'gene2': selected_gene_names[j],
                                    'expression_corelation': focal_expression_similarity_matrix[i, j], 
                                    'proximity_expression_product': expression_proximity_product[i, j]}, 
                                    index=[0])  # Create a DataFrame with a single row 
            bedpe = pd.concat([bedpe, new_row], ignore_index=True)


In [ ]:
print(bedpe.shape)

print(bedpe.head())

In [ ]:
# Save the bedpe dataframe to a tsv file without headers
bedpe.to_csv("expression_proximity.bedpe", sep="\t", index=False, header=False)